# PlantVillageVQA — MiniCPM5-1B + frozen SigLIP, gated cross-attention
Thin Kaggle driver. Add the **PlantVillageVQA** dataset as a Notebook input, enable **GPU T4 x2**.

**Before running:** in *Add-ons → Secrets*, create `GITHUB_TOKEN` (repo read) and `HF_TOKEN`
(HF read), and attach both to this notebook. The first cell clones the project from GitHub
using those secrets, so you don't need to upload the code manually. Then run top-to-bottom.

In [ ]:
# --- secrets + clone the project from GitHub ---
# In Kaggle: Add-ons -> Secrets -> add GITHUB_TOKEN and HF_TOKEN, then attach them to this notebook.
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
GITHUB_TOKEN = user_secrets.get_secret("GITHUB_TOKEN")
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

import os
REPO = "ryzewtf/GenAI_LAB_CA"
DST = "/kaggle/working/GenAI_LAB_CA"
if not os.path.isdir(DST):
    # token embedded in the URL so a private repo clones non-interactively
    os.system(f"git clone https://{GITHUB_TOKEN}@github.com/{REPO}.git {DST}")
%cd {DST}

# make HF_TOKEN available for gated model downloads (MiniCPM) done by from_pretrained
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
!ls

In [ ]:
# --- environment ---
# torch is preinstalled (CUDA build). NO flash-attn.
!pip -q install -U transformers datasets accelerate sentencepiece pillow pyyaml matplotlib huggingface_hub
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])
import torch
print('cuda', torch.cuda.is_available(), 'devices', torch.cuda.device_count())
print('capability', torch.cuda.get_device_capability(0), '(7,5) => T4/fp16')

In [ ]:
# --- download the frozen backbones separately (LLM + SigLIP) ---
# Pull once to /kaggle/working/models so training/caching load from local disk
# instead of re-fetching from the Hub. Surfaces any auth/network failure here.
from huggingface_hub import snapshot_download

LLM_LOCAL = snapshot_download(
    'openbmb/MiniCPM5-1B', local_dir='/kaggle/working/models/MiniCPM5-1B',
    token=os.environ['HF_TOKEN'])
VIS_LOCAL = snapshot_download(
    'google/siglip-base-patch16-224', local_dir='/kaggle/working/models/siglip',
    token=os.environ['HF_TOKEN'])
print('LLM    ->', LLM_LOCAL)
print('SigLIP ->', VIS_LOCAL)

In [ ]:
# --- point config at the Kaggle-mounted dataset + local models ---
import yaml, os
cfg_path = 'configs/default.yaml'
cfg = yaml.safe_load(open(cfg_path))

DATA_ROOT = '/kaggle/input/datasets/ryzewtf/plantvillagevqa/PlantVillageVQA'
assert os.path.isfile(os.path.join(DATA_ROOT, 'PlantVillageVQA.csv')), 'add the PlantVillageVQA dataset as a Notebook input'

cfg['data']['data_root'] = DATA_ROOT
cfg['data']['csv_name'] = 'PlantVillageVQA.csv'
cfg['data']['images_dirname'] = 'Images'
cfg['data']['prepared_dir'] = '/kaggle/working/prepared'
cfg['cache']['feature_dir'] = '/kaggle/working/cache'
cfg['train']['out_dir'] = '/kaggle/working/runs/exp1'
# load the frozen backbones from the local snapshots downloaded above
cfg['model']['llm_name'] = LLM_LOCAL
cfg['model']['vision_name'] = VIS_LOCAL
yaml.safe_dump(cfg, open(cfg_path, 'w'))

print('data_root  =', DATA_ROOT)
print('llm_name   =', cfg['model']['llm_name'])
print('vision     =', cfg['model']['vision_name'])
print('contents   =', os.listdir(DATA_ROOT))

In [ ]:
!python -m data.prepare --config configs/default.yaml --subset-size 30000

In [ ]:
!python -m data.cache_features --config configs/default.yaml

In [ ]:
# sanity: overfit a tiny slice (loss should fall, EM rise)
!python train.py --config configs/default.yaml --overfit 40 --grad-accum 1 --epochs 40

In [ ]:
!python train.py --config configs/default.yaml

In [ ]:
!python eval.py --config configs/default.yaml --ckpt /kaggle/working/runs/exp1/best.pt --split test
!python eval.py --config configs/default.yaml --ckpt /kaggle/working/runs/exp1/best.pt --split test --blind
!python analyze_gates.py --ckpt /kaggle/working/runs/exp1/best.pt --out /kaggle/working/runs/exp1/gates.png